# Read the JSON and print the results.

In [4]:
import json
import os
import pandas as pd

# Define the path to your leaderboard JSON
file_path = "master_leaderboard.json"

def display_leaderboard_as_table(path):
    if not os.path.exists(path):
        return f"Error: File not found at {path}"

    with open(path, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            return "Error: Failed to decode JSON."

    # 1. Flatten the nested 'params' into the main dictionary
    # This makes 'epochs' its own column
    flattened_data = []
    for entry in data:
        row = entry.copy()
        params = row.pop('params', {})  # Remove 'params' dict and extract content
        row['epochs'] = params.get('epochs', 'N/A')
        flattened_data.append(row)

    # 2. Create DataFrame
    df = pd.DataFrame(flattened_data)

    # 3. Clean up and Sort
    # Selecting and ordering the columns you want
    cols = ['dataset', 'features', 'model', 'epochs', 'test_f1', 'test_acc']
    df = df[cols]
    
    # Sort: Dataset A-Z, then Test F1 High to Low
    df = df.sort_values(by=['dataset', 'test_f1'], ascending=[True, False])

    # 4. Format numbers for a cleaner look
    df['test_f1'] = df['test_f1'].map('{:.4f}'.format)
    df['test_acc'] = df['test_acc'].map('{:.4f}'.format)

    # 5. Output
    # This will print like the Rating | Count table you showed
    print(df.to_string(index=False))

# Execute
if __name__ == "__main__":
    display_leaderboard_as_table(file_path)


       dataset                              features     model  epochs test_f1 test_acc
Amazon Ratings                  amazon-mpnet-base-v2 GraphSAGE     200  0.5041   0.5493
Amazon Ratings                                 SBERT GraphSAGE     100  0.4763   0.5395
Amazon Ratings                              FastText GraphSAGE     100  0.4403   0.5172
Amazon Ratings                                 SBERT     H2GCN     100  0.4385   0.4987
Amazon Ratings                              FastText     H2GCN     100  0.4373   0.5009
Amazon Ratings                  amazon-mpnet-base-v2 GraphSAGE     100  0.4206   0.5081
Amazon Ratings                                 SBERT     H2GCN     100  0.4185   0.4799
Amazon Ratings                  amazon-mpnet-base-v2       GAT     200  0.4175   0.4915
Amazon Ratings                  amazon-mpnet-base-v2     H2GCN     100  0.4160   0.4850
Amazon Ratings                                 SBERT       GCN     100  0.3968   0.4658
Amazon Ratings                  

In [5]:
import json
import os
import pandas as pd

file_path = "master_leaderboard.json"

def display_best_combinations(path):
    if not os.path.exists(path):
        return f"Error: File not found at {path}"

    with open(path, 'r') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            return "Error: Failed to decode JSON."

    # 1. Flatten the nested 'params'
    flattened_data = []
    for entry in data:
        row = entry.copy()
        params = row.pop('params', {})
        row['epochs'] = params.get('epochs', 'N/A')
        flattened_data.append(row)

    df = pd.DataFrame(flattened_data)

    # 2. Select the BEST result for each (Dataset, Features, Model) combo
    # We group by those three columns and find the index of the maximum 'test_f1'
    idx = df.groupby(['dataset', 'features', 'model'])['test_f1'].idxmax()
    best_df = df.loc[idx].copy()

    # 3. Sort by Dataset and Features (as requested)
    best_df = best_df.sort_values(by=['dataset', 'features'], ascending=[True, True])

    # 4. Final Formatting
    cols = ['dataset', 'features', 'model', 'epochs', 'test_f1', 'test_acc']
    best_df = best_df[cols]
    
    best_df['test_f1'] = best_df['test_f1'].map('{:.4f}'.format)
    best_df['test_acc'] = best_df['test_acc'].map('{:.4f}'.format)

    # Display like a clean dataframe table
    print("--- Best Results per Combination ---")
    print(best_df.to_string(index=False))

if __name__ == "__main__":
    display_best_combinations(file_path)


--- Best Results per Combination ---
       dataset                              features     model  epochs test_f1 test_acc
Amazon Ratings                              FastText       GAT     100  0.3283   0.4595
Amazon Ratings                              FastText       GCN     100  0.3625   0.4623
Amazon Ratings                              FastText      GCN2     400  0.3613   0.4605
Amazon Ratings                              FastText GraphSAGE     100  0.4403   0.5172
Amazon Ratings                              FastText     H2GCN     100  0.4373   0.5009
Amazon Ratings                                 SBERT       GAT     100  0.3621   0.4748
Amazon Ratings                                 SBERT       GCN     100  0.3968   0.4658
Amazon Ratings                                 SBERT      GCN2     400  0.3604   0.4589
Amazon Ratings                                 SBERT GraphSAGE     100  0.4763   0.5395
Amazon Ratings                                 SBERT     H2GCN     100  0.4385   0.

In [12]:
import json
import os
import pandas as pd

file_path = "master_leaderboard.json"

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

def create_comparison_tables(path):
    if not os.path.exists(path):
        return f"Error: File not found at {path}"

    with open(path, 'r') as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    # Clean up model names
    df['model'] = df['model'].astype(str).str.replace(
        r'.*Custom BERT.*Roman Empire.*', 'Custom BERT', regex=True, case=False
    )

    def get_dataset_view(dataset_name):
        subset = df[df['dataset'] == dataset_name].copy()
        if subset.empty: return None

        # Pivot to get max values
        pivot_f1 = subset.pivot_table(index='model', columns='features', values='test_f1', aggfunc='max')
        pivot_acc = subset.pivot_table(index='model', columns='features', values='test_acc', aggfunc='max')

        view = pd.DataFrame(index=pivot_f1.index, columns=pivot_f1.columns, dtype=object)

        for col in pivot_f1.columns:
            for idx in pivot_f1.index:
                f1, acc = pivot_f1.loc[idx, col], pivot_acc.loc[idx, col]
                if pd.isna(f1):
                    view.loc[idx, col] = "      -      "
                else:
                    view.loc[idx, col] = f"{f1:.4f} / {acc:.4f}"
        
        view.index.name = "MODEL"
        return view

    # --- THE KEY CHANGE ---
    # Automatically get every unique dataset name present in your JSON
    unique_datasets = sorted(df['dataset'].unique())

    for ds in unique_datasets:
        print("\n" + "="*80)
        print(f"  {ds.upper()} LEADERBOARD (Best F1 / Acc)")
        print("="*80)
        view = get_dataset_view(ds)
        if view is not None:
            print(view.sort_index(axis=1).to_string())

if __name__ == "__main__":
    create_comparison_tables(file_path)



  AMAZON RATINGS LEADERBOARD (Best F1 / Acc)
features          FastText            SBERT amazon-mpnet-base-v2
MODEL                                                           
GAT        0.3283 / 0.4595  0.3621 / 0.4748      0.4175 / 0.4915
GCN        0.3625 / 0.4623  0.3968 / 0.4658      0.3750 / 0.4701
GCN2       0.3613 / 0.4605  0.3604 / 0.4589      0.3543 / 0.4552
GraphSAGE  0.4403 / 0.5172  0.4763 / 0.5395      0.5041 / 0.5493
H2GCN      0.4373 / 0.5009  0.4385 / 0.4987      0.4160 / 0.4850

  CORA LEADERBOARD (Best F1 / Acc)
features               PyG
MODEL                     
GAT        0.8404 / 0.8510
GCN        0.8595 / 0.8730
GCN2       0.8350 / 0.8440
GraphSAGE  0.8502 / 0.8590
H2GCN      0.8487 / 0.8610

  CORNELL LEADERBOARD (Best F1 / Acc)
features               PyG
MODEL                     
GAT        0.3438 / 0.4324
GCN        0.3085 / 0.4054
GraphSAGE  0.4246 / 0.5946
H2GCN      0.6601 / 0.7297

  PUBMED LEADERBOARD (Best F1 / Acc)
features               PyG
MODEL   